# Robust Time-Series Preprocessing & DM=0 RFI Masking

Demonstrates:
- `PreprocessOptions` and `preprocess_timeseries` (multi-scale block flagging, outlier clipping, heteroscedastic inverse-variance weights `(ts_e, ts_v)`, and birdie zapping).
- `generate_dm0_rfi_mask` and `apply_dm0_rfi_mask` / `TimeSeries.apply_dm0_rfi_mask`.
- `ObservationMetadata` propagation.

In [1]:
import numpy as np

from pyloki.io.preprocess import (
    Birdie,
    ObservationMetadata,
    PreprocessOptions,
    apply_dm0_rfi_mask,
    generate_dm0_rfi_mask,
    preprocess_timeseries,
)
from pyloki.io.timeseries import TimeSeries

rng = np.random.default_rng(42)
dt = 1e-3
n = 16384
t = np.arange(n, dtype=np.float64) * dt

# Synthetic raw DM=0 time series with time-varying noise, a 50 Hz mains birdie,
# a broadband RFI burst, and an isolated spike.
sigma_t = np.where(t < 8.0, 1.0, 2.5)
raw_dm0 = sigma_t * rng.standard_normal(n) + 4.0 * np.sin(2.0 * np.pi * 50.0 * t)
raw_dm0[4000:4128] += 12.0
raw_dm0[9000] += 35.0

In [2]:
opts = PreprocessOptions(
    method="robust",
    filter_window=1.0,
    window_blocks=16,
    block_scales=(0.064, 0.128),
    block_sigma=5.0,
    clip_sigma=6.0,
    birdies=(Birdie(freq=50.0, width=0.5),),
    zap_mode="noise",
)

# 1. Preprocess DM=0 and build a reusable DM=0 RFI mask
dm0_e, dm0_v, dm0_rep = preprocess_timeseries(raw_dm0, dt, options=opts)
dm0_mask = generate_dm0_rfi_mask(dm0_e, dm0_v, dt, t_sigma=5.0, f_sigma=8.0)
print(
    f"DM=0 mask flagged {dm0_mask.mask_t.size}/{n} time samples "
    f"and {dm0_mask.mask_f.size} spectral bins"
)

DM=0 mask flagged 194/16384 time samples and 7 spectral bins


In [3]:
# 2. Preprocess a dedispersed DM > 0 time series and apply the DM=0 mask
raw_dm = sigma_t * rng.standard_normal(n)
ts_e, ts_v, report = preprocess_timeseries(raw_dm, dt, options=opts)
meta = ObservationMetadata(dm=120.5, start_mjd=60000.5, source="SYNTH_PSR")
ts = TimeSeries(ts_e, ts_v, dt, metadata=meta)

ts_masked = ts.apply_dm0_rfi_mask(dm0_mask, seed=7)
ts_e2, ts_v2 = apply_dm0_rfi_mask(ts_e, ts_v, dm0_mask, seed=7)

print(
    f"Preprocess report: n_clipped={report.n_clipped}, "
    f"zero_weight_frac={report.zero_weight_fraction:.4f}"
)
print(
    f"Masked zero-weight samples: {int(np.sum(ts_masked.ts_v == 0.0))} "
    f"(direct helper: {int(np.sum(ts_v2 == 0.0))})"
)

Preprocess report: n_clipped=0, zero_weight_frac=0.0078
Masked zero-weight samples: 321 (direct helper: 321)
